# World Cup 2022 Prediction — Refined

This version fixes the original project’s main methodological issues: draws are modeled explicitly, features are leakage-safe, validation is walk-forward/time-based, models are compared on probabilistic metrics, and the final tournament is simulated with 10,000 Monte Carlo runs.

**Run `refined_project.py` for the complete pipeline.**

In [ ]:
from refined_project import *

# Load and validate the original datasets.
hist, groups, wc_matches = prepare_data()
print(f"Historical matches: {len(hist):,}")
print(f"Date range: {hist['Date'].min().date()} → {hist['Date'].max().date()}")
print(hist['Result'].value_counts(normalize=True).round(3))

## 1. Leakage-safe feature engineering

Every feature is generated from the team state **before** the match: ELO, last-five form, win rate, recent goals and goal difference. The current match is only used to update the state after the features have been recorded.

In [ ]:
featured = build_pre_match_features(hist)
featured[FEATURES].head()

## 2. Walk-forward validation

No shuffled K-fold CV is used. Each validation block is strictly later than its training data. We report **accuracy, log loss and Brier score**.

In [ ]:
validation, oof = walk_forward_evaluation(featured)
summary = (
    validation.groupby('model')[['accuracy','log_loss','brier']]
    .mean()
    .sort_values('log_loss')
)
summary

In [ ]:
log_weight, xgb_weight = select_ensemble_weights(oof)
print(f"Selected probability blend: Logistic={log_weight:.2f}, XGBoost={xgb_weight:.2f}")

## 3. Train final pre-tournament models

The final models are trained only on historical matches available before the 2022 World Cup. The 2022 World Cup results are never used for training.

In [ ]:
log_model, xgb_model, _, _ = train_final_models(featured, log_weight, xgb_weight)
home_goal_model, away_goal_model = train_goal_models(featured)
states = team_state_as_of(featured)

# Example pre-tournament prediction
p = match_probabilities(
    'Brazil', 'Argentina', states,
    log_model, xgb_model, log_weight, xgb_weight, host=False
)
print(dict(zip(CLASS_NAMES, p.round(3))))

## 4. 10,000-run Monte Carlo tournament simulation

The simulation models all 48 group-stage games, applies the official 2022 knockout bracket, resolves knockout draws with a penalty shootout, and reports the probability of reaching each stage.

In [ ]:
group_matches = build_group_matches(groups, wc_matches)
probabilities = run_simulations(
    groups, group_matches, states,
    (log_model, xgb_model, log_weight, xgb_weight, home_goal_model, away_goal_model),
    n=N_SIMULATIONS
)

cols = ['Team','R16 Probability','QF Probability','SF Probability','Final Probability','Champion Probability']
probabilities[cols].head(20)

In [ ]:
# Save outputs for GitHub / resume writing.
save_results(validation, probabilities, groups, states)
print('Saved to:', RESULTS_DIR)

## 5. What to put on the resume after running

Do **not** copy the placeholder numbers below. Replace them with the outputs from `results/model_summary.csv` and `results/world_cup_probabilities.csv`.

- Built a leakage-safe multiclass football prediction pipeline using ELO, recent form and goal-difference features across **15k+ international matches**; evaluated Logistic Regression, Random Forest and XGBoost using walk-forward validation.
- Developed a **10,000-run Monte Carlo World Cup simulator** incorporating group-stage standings, knockout progression and probabilistic match outcomes; estimated team-level probabilities for R16, QF, SF, Final and Champion.
- Achieved **[XX]% out-of-time accuracy**, **[X.XXX] log loss** and **[X.XXX] Brier score** on future match blocks; selected a Logistic/XGBoost probability ensemble using validation performance.